# Ensemble pipeline on Kaggle: YOLOv8s-P2 + Faster R-CNN → SAHI → WBF → submission

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, Add Input → the competition data.

Pick `STAGE` in the second code cell, then **Save Version → Save & Run All** (runs in the background, up to 12 h):

| Session | `STAGE` | What happens | Time |
|---|---|---|---|
| 1 | `"smoke"` | whole pipeline on 2 clips, 2 epochs (`--debug`) | ~20 min |
| 2, 3, … | `"train"` | tiles, then YOLO on GPU 0 and Faster R-CNN on GPU 1 **in parallel**; stops after `TRAIN_HOURS` | ≤ 12 h each |
| last | `"infer"` | SAHI on val + test, WBF grid on val, `submission.csv` | ~3–4 h |
| HR 1 | `"train_hr"` | full-frame YOLOv8s at 1280 (`src.train_yolo_hr`, both T4 via DDP): speed benchmark → `frame_step` → time-boxed training (`yolo_hr.time` = 10 h) | ≤ 10.5 h |
| HR 2 | `"infer_hr"` | `yolo_hr` on full val + test frames → `work/preds/yolo_hr_{val,test}.parquet` + its val metrics row | ~10 min |

**Continuing:** a session only sees its own files. Before the next session, Add Input → *Your Work* → the
output of this notebook's previous version; the restore cell copies `runs/` and `work/` back and training
resumes from the last epoch. Repeat `"train"` until it prints that both models are `done`, then run `"infer"`.

All settings (epochs, batch, grid) are in `configs/pipeline.yaml` of the repository.

`train_hr` / `infer_hr` never tile, train or run YOLO-P2 / Faster R-CNN: their predictions come back with `work/preds/` from the attached previous version. If `train_hr` is cut off, run it again with that version attached: it resumes from `runs/yolo_hr/weights/last.pt`.


In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path("/tmp/repo")  # outside /kaggle/working: not saved as output
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/LiliAlania/drone-multi-object-detection.git", str(REPO)],
                   check=True)
os.chdir(REPO)
# torch/torchvision are preinstalled on Kaggle; pin only what the pipeline adds
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.208", "sahi==0.14.0",
                "ensemble-boxes==1.0.10", "pycocotools"], check=True)
!nvidia-smi -L

In [ ]:
STAGE = "smoke"      # "smoke" | "train" | "infer" | "train_hr" | "infer_hr"
TRAIN_HOURS = 11.0   # stop training before Kaggle's 12 h limit; the next session resumes

INPUT, OUT = Path("/kaggle/input"), Path("/kaggle/working")
found = [p for depth in range(1, 5) for p in INPUT.glob("*/" * depth + "val/annotations_coco.json")]
assert found, "Competition data not found: Add Input -> the competition"
DATA = found[0].parent.parent
assert (DATA / "test_images.csv").is_file(), f"no test_images.csv in {DATA}"
os.environ.update(PIPELINE_ENV="kaggle", PIPELINE_DATA_ROOT=str(DATA))
DEBUG = ["--debug"] if STAGE == "smoke" else []
SUB = "debug" if DEBUG else ""
RUNS, WORK = OUT / "runs" / SUB, OUT / "work" / SUB
print("data:", DATA, "| stage:", STAGE)

In [ ]:
# restore runs/ and work/ from earlier versions of this notebook attached as input
for previous in sorted(p.parent for p in INPUT.glob("*/runs") if p.is_dir()):
    for name in ("runs", "work"):
        if (previous / name).is_dir():
            shutil.copytree(previous / name, OUT / name, dirs_exist_ok=True)
            print(f"restored {previous / name}")

In [ ]:
def run(*args):
    """Run `python -m <args>` and stream its output (only the final state of progress-bar lines)."""
    cmd = [sys.executable, "-m", *args, *DEBUG]
    print("$", " ".join(cmd[2:]), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        line = line.rstrip("\n").split("\r")[-1]
        if line.strip():
            print(line, flush=True)
    if process.wait():
        raise RuntimeError(f"{args[0]} failed with exit code {process.returncode}")


def status(model):
    """'new', 'resume' or 'done' from the model's last checkpoint."""
    import torch

    last = RUNS / "yolo_p2" / "weights" / "last.pt" if model == "yolo" else RUNS / "frcnn" / "last.pt"
    if not last.is_file():
        return "new"
    state = torch.load(last, map_location="cpu", weights_only=False)
    if model == "yolo":
        return "done" if state.get("epoch", -1) == -1 else "resume"  # Ultralytics sets -1 when training ends
    return "done" if state["epoch"] >= state["epochs"] else "resume"

In [ ]:
if STAGE in ("smoke", "train"):
    import torch

    jobs = {"yolo": "src.train_yolo_p2", "frcnn": "src.train_frcnn_tiles"}
    todo = {m: s for m in jobs if (s := status(m)) != "done"}
    print("status:", {m: status(m) for m in jobs})
    if todo:
        run("src.prepare_tiles", "--force")  # tiles are not kept between sessions; same seed -> same tiles
        logs = OUT / "logs"
        logs.mkdir(exist_ok=True)
        gpus = max(torch.cuda.device_count(), 1)
        processes = {}
        for index, (model, state) in enumerate(todo.items()):
            cmd = [sys.executable, "-m", jobs[model], *DEBUG] + (["--resume"] if state == "resume" else [])
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(index % gpus)}  # one model per T4
            processes[model] = subprocess.Popen(cmd, stdout=open(logs / f"{model}.log", "a"),
                                                stderr=subprocess.STDOUT, env=env)
            print(f"{model}: {state} on GPU {index % gpus}, log {logs / f'{model}.log'}")
        deadline, last_report = time.time() + TRAIN_HOURS * 3600, 0.0
        while any(p.poll() is None for p in processes.values()):
            time.sleep(30)
            if time.time() - last_report > 1800:  # progress every 30 min
                last_report = time.time()
                for model in processes:
                    text = (logs / f"{model}.log").read_text(errors="ignore").replace("\r", "\n").strip()
                    print(f"[{time.strftime('%H:%M')}] {model}: {text.splitlines()[-1][:160] if text else '...'}")
            if time.time() > deadline:
                print("TRAIN_HOURS reached: stopping; the next session resumes from last.pt")
                for p in processes.values():
                    p.terminate()
                for p in processes.values():
                    p.wait()
        for model, p in processes.items():
            print(f"{model}: exit code {p.returncode} (log: {logs / f'{model}.log'})")
    print("status:", {m: status(m) for m in jobs})

In [ ]:
if STAGE in ("smoke", "infer"):
    not_done = [m for m in ("yolo", "frcnn") if status(m) != "done"]
    assert not not_done, f"{not_done} still training: run STAGE='train' again first"
    run("src.run_pipeline", "--from", "infer")
    print((WORK / "ensemble" / "comparison.md").read_text())
    target = OUT / ("submission_debug.csv" if DEBUG else "submission.csv")
    shutil.copy(WORK / "submission.csv", target)
    print("saved", target)

In [ ]:
if STAGE == "train_hr":  # high-res full-frame YOLO only; the ensemble models are not touched
    import signal
    import torch

    last = RUNS / "yolo_hr" / "weights" / "last.pt"
    state = ("new" if not last.is_file() else  # Ultralytics sets epoch -1 when training ends
             "done" if torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1 else "resume")
    print("yolo_hr:", state)
    if state != "done":
        logs = OUT / "logs"
        logs.mkdir(exist_ok=True)
        log = logs / "yolo_hr.log"
        cmd = [sys.executable, "-m", "src.train_yolo_hr"] + (["--resume"] if state == "resume" else [])
        # own process group: the deadline kill also reaches the DDP workers
        process = subprocess.Popen(cmd, stdout=open(log, "a"), stderr=subprocess.STDOUT, start_new_session=True)
        print(f"{' '.join(cmd[2:])}: log {log}")
        deadline, last_report = time.time() + TRAIN_HOURS * 3600, time.time()
        while process.poll() is None:
            time.sleep(30)
            if time.time() - last_report > 1800:  # progress every 30 min
                last_report = time.time()
                text = log.read_text(errors="ignore").replace("\r", "\n").strip()
                print(f"[{time.strftime('%H:%M')}] yolo_hr: {text.splitlines()[-1][:160] if text else '...'}")
            if time.time() > deadline:
                print("TRAIN_HOURS reached: stopping; the next session resumes from last.pt")
                os.killpg(process.pid, signal.SIGTERM)
                process.wait()
        print(f"yolo_hr: exit code {process.returncode}")
        lines = [l for l in log.read_text(errors="ignore").replace("\r", "\n").splitlines()
                 if any(k in l for k in ("Measured", "frame_step", "class ids", "Training ", "WARNING", "failed",
                                         "epochs completed", "Error"))]
        print("\n".join(lines))
    results = RUNS / "yolo_hr" / "results.csv"
    if results.is_file():
        import pandas as pd

        table = pd.read_csv(results)
        table.columns = table.columns.str.strip()
        print(table.tail(5).to_string(index=False))


In [ ]:
if STAGE == "infer_hr":  # yolo_hr predictions only; YOLO-P2 / Faster R-CNN preds stay as restored
    assert (RUNS / "yolo_hr" / "weights" / "best.pt").is_file(), "no yolo_hr weights: run STAGE='train_hr' first"
    for split in ("val", "test"):  # val also prints the comparison.md row
        run("src.infer_sahi", "--model", "yolo_hr", "--split", split)
    for path in sorted((WORK / "preds").glob("*.parquet")):
        print("saved", path)


In [ ]:
# keep the saved output small: tiles are rebuilt each session
shutil.rmtree(WORK / "tiles", ignore_errors=True)
shutil.rmtree(WORK / "yolo_hr", ignore_errors=True)  # full-frame training copies, same
!du -sh /kaggle/working/* 2>/dev/null